# Byte: Socratic C++ tutor (Gemma 3 1B + LoRA)
Pipeline test on the 24-example Socratic seed set (hints first, answers on request).

**Before you start:** Runtime > Change runtime type > **T4 GPU**. Then run the cells top to bottom.
You will upload `cpp_socratic_seed.jsonl` in Step 2.

## Step 1: Install Unsloth

In [ ]:
%%capture
!pip install unsloth

## Step 2: Upload and load the dataset

In [ ]:
from google.colab import files
from datasets import load_dataset

uploaded = files.upload()          # choose cpp_socratic_seed.jsonl
data_file = list(uploaded.keys())[0]

ds = load_dataset("json", data_files=data_file, split="train")
print(ds)

SYS = ds[0]["messages"][0]["content"]   # same system prompt used in training
print("\nSystem prompt:\n", SYS)

## Step 3: Load Gemma 3 1B (4-bit)

In [ ]:
from unsloth import FastModel
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-3-1b-it",
    max_seq_length = 2048,
    load_in_4bit = True,
    full_finetuning = False,
)

from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(tokenizer, chat_template = "gemma-3")

## Step 4: Helpers and BEFORE test (base model, no training yet)
The test messages use the slash commands. Save the base model's answers for your writeup.

In [ ]:
def fold_system(messages):
    # Gemma has no system role, so put the system prompt at the start of the first user message.
    if messages[0]["role"] == "system":
        first = {"role": "user", "content": messages[0]["content"] + "\n\n" + messages[1]["content"]}
        return [first] + messages[2:]
    return messages

def ask(user_msg, history = None, max_new_tokens = 300):
    # history = earlier turns as [{"role": "user"/"assistant", "content": ...}, ...] for multi-turn tests
    msgs = fold_system(
        [{"role": "system", "content": SYS}] + (history or []) + [{"role": "user", "content": user_msg}]
    )
    text = tokenizer.apply_chat_template(msgs, tokenize = False, add_generation_prompt = True)
    inputs = tokenizer([text], add_special_tokens = False, return_tensors = "pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens = max_new_tokens,
                         do_sample = True, temperature = 0.7, top_p = 0.95, top_k = 64)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens = True)

# New messages that are NOT in the training set
TESTS = [
    "/debug\nint x = 10;\nint *p;\n*p = 5;\ncout << x;",
    "/explain\nWhat is a reference?",
    "Write a program that checks if a number is prime.",
    "/quiz\nvectors",
]

for t in TESTS:
    print("=" * 60); print(t); print("-" * 60)
    print(ask(t))

## Step 5: Format the data for Gemma

In [ ]:
def to_text(example):
    msgs = fold_system(example["messages"])
    text = tokenizer.apply_chat_template(msgs, tokenize = False, add_generation_prompt = False)
    return {"text": text.removeprefix("<bos>")}

train_ds = ds.map(to_text)
print(train_ds[0]["text"][:700])

## Step 6: Add LoRA adapters

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

## Step 7: Train
24 examples x 8 epochs is about 24 steps, a few minutes on a T4.

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        num_train_epochs = 8,
        warmup_steps = 3,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none",
    ),
)

# Learn only from Byte's replies, not from the user's code
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",
    response_part = "<start_of_turn>model\n",
)

stats = trainer.train()

## Step 8: AFTER test (same messages as Step 4)
Good signs: one guiding question, no corrected code after /debug, and a direct fix after /solution.

In [ ]:
for t in TESTS:
    print("=" * 60); print(t); print("-" * 60)
    print(ask(t))

# Multi-turn check: does it give a hint first, then the solution when asked?
h = [
    {"role": "user", "content": TESTS[0]},
    {"role": "assistant", "content": ask(TESTS[0])},
]
print("=" * 60); print("/hint"); print("-" * 60)
print(ask("/hint", history = h))

## Step 9: Save the LoRA adapter to Google Drive (safe backup)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
os.makedirs("/content/drive/MyDrive/byte", exist_ok = True)
model.save_pretrained("/content/drive/MyDrive/byte/lora_adapter")
tokenizer.save_pretrained("/content/drive/MyDrive/byte/lora_adapter")
print("Adapter saved.")

## Step 10: Export to GGUF (q4_k_m) for PocketPal and Termux
This builds llama.cpp inside Colab and can take several minutes.
The argument name for the quantization has changed between Unsloth versions, so the cell tries both.

In [ ]:
try:
    model.save_pretrained_gguf("gemma-byte", tokenizer, quantization_method = "q4_k_m")
except TypeError:
    model.save_pretrained_gguf("gemma-byte", quantization_type = "Q4_K_M")

import glob, shutil
ggufs = glob.glob("gemma-byte*/**/*.gguf", recursive = True) + glob.glob("*.gguf")
print("Found:", ggufs)
for g in ggufs:
    shutil.copy(g, "/content/drive/MyDrive/byte/")
print("Copied to Google Drive > byte")

## Done
Download the `.gguf` from Google Drive > byte, load it in PocketPal, and create a Pal with the same system prompt
(printed in Step 2). Compare the AFTER answers with your BEFORE answers for the writeup.